# Processing of extracted trajectory data
- In this notebook we use the output of the trajectory data extraction
- We compute some additional features
- We perform a train test split for the new data and the old dataset and save it

In [1]:
import sys
import os
sys.path.append(os.path.abspath("../.."))

from processing.filtering import filter_for_players

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import euclidean

from itertools import combinations

pd.set_option('display.max_columns', None)

In [2]:
# Config
basket_x = 89.25
basket_y = 25

do_filter_for_players = False

# Data loading and filtering

In [3]:
df = pd.read_csv('merged_tracking_events_v3.csv', index_col=0)
df.head()

,shooter_slot,shooter_x,shooter_y,shooter_team_id,shot_angle,distance_to_basket_tracking,nearest_defender_dist,avg_defender_dist,defenders_within_3ft,defenders_within_5ft,defenders_within_7ft,offensive_spacing_area,shooter_speed,player1_speed,player2_speed,player3_speed,player4_speed,player5_speed,player6_speed,player7_speed,player8_speed,player9_speed,player10_speed,defender_closing_speed,ball_height,ball_speed,ball_xy_speed,ACTION_TYPE,EVENTTIME,EVENT_TYPE,GAME_DATE,GAME_EVENT_ID,GAME_ID,GRID_TYPE,HTM,LOC_X,LOC_Y,MINUTES_REMAINING,PERIOD,PLAYER_ID,PLAYER_NAME,QUARTER,SECONDS_REMAINING,SHOT_ATTEMPTED_FLAG,SHOT_DISTANCE,SHOT_MADE_FLAG,SHOT_TIME,SHOT_TYPE,SHOT_ZONE_AREA,SHOT_ZONE_BASIC,SHOT_ZONE_RANGE,TEAM_ID,TEAM_NAME,VTM,tracking_game_clock,tracking_game_clock_old,event_GAME_ID,event_EVENTNUM,event_EVENTMSGTYPE,event_EVENTMSGACTIONTYPE,event_PERIOD,event_WCTIMESTRING,event_PCTIMESTRING,event_HOMEDESCRIPTION,event_NEUTRALDESCRIPTION,event_VISITORDESCRIPTION,event_SCORE,event_SCOREMARGIN,event_PERSON1TYPE,event_PLAYER1_ID,event_PLAYER1_NAME,event_PLAYER1_TEAM_ID,event_PLAYER1_TEAM_CITY,event_PLAYER1_TEAM_NICKNAME,event_PLAYER1_TEAM_ABBREVIATION,event_PERSON2TYPE,event_PLAYER2_ID,event_PLAYER2_NAME,event_PLAYER2_TEAM_ID,event_PLAYER2_TEAM_CITY,event_PLAYER2_TEAM_NICKNAME,event_PLAYER2_TEAM_ABBREVIATION,event_PERSON3TYPE,event_PLAYER3_ID,event_PLAYER3_NAME,event_PLAYER3_TEAM_ID,event_PLAYER3_TEAM_CITY,event_PLAYER3_TEAM_NICKNAME,event_PLAYER3_TEAM_ABBREVIATION,ball_x,ball_y,ball_z,player1_team_id,player1_id,player1_x,player1_y,player2_team_id,player2_id,player2_x,player2_y,player3_team_id,player3_id,player3_x,player3_y,player4_team_id,player4_id,player4_x,player4_y,player5_team_id,player5_id,player5_x,player5_y,player6_team_id,player6_id,player6_x,player6_y,player7_team_id,player7_id,player7_x,player7_y,player8_team_id,player8_id,player8_x,player8_y,player9_team_id,player9_id,player9_x,player9_y,player10_team_id,player10_id,player10_x,player10_y,shooter_x_t0,shooter_y_t0,defender1_dx_t0,defender1_dy_t0,defender1_dist_t0,defender2_dx_t0,defender2_dy_t0,defender2_dist_t0,defender3_dx_t0,defender3_dy_t0,defender3_dist_t0,defender4_dx_t0,defender4_dy_t0,defender4_dist_t0,defender5_dx_t0,defender5_dy_t0,defender5_dist_t0,attacker1_dx_t0,attacker1_dy_t0,attacker1_dist_t0,attacker2_dx_t0,attacker2_dy_t0,attacker2_dist_t0,attacker3_dx_t0,attacker3_dy_t0,attacker3_dist_t0,attacker4_dx_t0,attacker4_dy_t0,attacker4_dist_t0,shooter_x_t1,shooter_y_t1,defender1_dx_t1,defender1_dy_t1,defender1_dist_t1,defender2_dx_t1,defender2_dy_t1,defender2_dist_t1,defender3_dx_t1,defender3_dy_t1,defender3_dist_t1,defender4_dx_t1,defender4_dy_t1,defender4_dist_t1,defender5_dx_t1,defender5_dy_t1,defender5_dist_t1,attacker1_dx_t1,attacker1_dy_t1,attacker1_dist_t1,attacker2_dx_t1,attacker2_dy_t1,attacker2_dist_t1,attacker3_dx_t1,attacker3_dy_t1,attacker3_dist_t1,attacker4_dx_t1,attacker4_dy_t1,attacker4_dist_t1,shooter_x_t2,shooter_y_t2,defender1_dx_t2,defender1_dy_t2,defender1_dist_t2,defender2_dx_t2,defender2_dy_t2,defender2_dist_t2,defender3_dx_t2,defender3_dy_t2,defender3_dist_t2,defender4_dx_t2,defender4_dy_t2,defender4_dist_t2,defender5_dx_t2,defender5_dy_t2,defender5_dist_t2,attacker1_dx_t2,attacker1_dy_t2,attacker1_dist_t2,attacker2_dx_t2,attacker2_dy_t2,attacker2_dist_t2,attacker3_dx_t2,attacker3_dy_t2,attacker3_dist_t2,attacker4_dx_t2,attacker4_dy_t2,attacker4_dist_t2,shooter_x_t3,shooter_y_t3,defender1_dx_t3,defender1_dy_t3,defender1_dist_t3,defender2_dx_t3,defender2_dy_t3,defender2_dist_t3,defender3_dx_t3,defender3_dy_t3,defender3_dist_t3,defender4_dx_t3,defender4_dy_t3,defender4_dist_t3,defender5_dx_t3,defender5_dy_t3,defender5_dist_t3,attacker1_dx_t3,attacker1_dy_t3,attacker1_dist_t3,attacker2_dx_t3,attacker2_dy_t3,attacker2_dist_t3,attacker3_dx_t3,attacker3_dy_t3,attacker3_dist_t3,attacker4_dx_t3,attacker4_dy_t3,attacker4_dist_t3,shooter_x_t4,shooter_y_t4,defender1_dx_t4,defender1_dy_t4,defender1_dist_t4,defender2_dx_t4,defender2_dy_t4,defender2_d

In [4]:
df.info(max_cols=200, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 84465 entries, 0 to 84464
Columns: 306 entries, shooter_slot to attacker4_dist_t5
dtypes: float64(255), int64(22), object(29)
memory usage: 197.8+ MB


In [5]:
df = df.dropna(subset=['shooter_x', 'shooter_y', 'shooter_speed', 'defender_closing_speed'])
#df = df.dropna(subset=['shooter_x', 'shooter_y', 'defender_closing_speed'])

In [6]:
df.info(max_cols=200, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 84103 entries, 0 to 84464
Columns: 306 entries, shooter_slot to attacker4_dist_t5
dtypes: float64(255), int64(22), object(29)
memory usage: 197.0+ MB


In [7]:
if do_filter_for_players:
    df = filter_for_players(df)
    df.info(max_cols=200, show_counts=True)

In [8]:
df['PLAYER_NAME'].unique()

array(['Marvin Williams', 'Nicolas Batum', 'Luis Scola', 'Cory Joseph',
       'Jonas Valanciunas', 'Terrence Ross', 'Frank Kaminsky',
       'DeMar DeRozan', 'Patrick Patterson', 'Brian Roberts',
       'Bismack Biyombo', 'Tyler Hansbrough', 'Kemba Walker',
       'Jeremy Lamb', 'DeMarre Carroll', 'Kyle Lowry', 'Cody Zeller',
       'PJ Hairston', 'Wesley Matthews', 'JaVale McGee',
       'Hassan Whiteside', 'Dirk Nowitzki', 'Gerald Green',
       'Zaza Pachulia', 'Deron Williams', 'Tyler Johnson', 'Dwyane Wade',
       'Udonis Haslem', 'Dwight Powell', 'Goran Dragic', 'Luol Deng',
       'Jeremy Evans', 'Raymond Felton', 'Charlie Villanueva',
       'Chandler Parsons', 'John Jenkins', 'J.J. Barea', 'Chris Bosh',
       'Chris Andersen', 'Jerian Grant', 'Kevin Seraphin', 'Bobby Portis',
       'Taj Gibson', 'Tony Snell', 'Aaron Brooks', 'Jimmy Butler',
       'Doug McDermott', 'Nikola Mirotic', 'Langston Galloway',
       'Derrick Williams', 'Kirk Hinrich', 'Sasha Vujacic', 'Robin Lop

# Generate additional features

In [9]:
# Extract player data
p_x = df[[f"player{i}_x" for i in range(1, 11)]].to_numpy()
p_y = df[[f"player{i}_y" for i in range(1, 11)]].to_numpy()
p_ids = df[[f"player{i}_id" for i in range(1, 11)]].to_numpy()
p_teams = df[[f"player{i}_team_id" for i in range(1, 11)]].to_numpy()

# shooter data
s_x = df["shooter_x"].to_numpy()[:, np.newaxis]
s_y = df["shooter_y"].to_numpy()[:, np.newaxis]
s_id = df["PLAYER_ID"].to_numpy()[:, np.newaxis]
s_team = df["TEAM_ID"].to_numpy()[:, np.newaxis]

# helper: distance between shooter and all players
dx = p_x - s_x
dy = p_y - s_y
dist_to_players = np.sqrt(dx**2 + dy**2)

## Helpers

In [10]:
def get_nearest_teammate_dist(dist_to_players, p_teams, s_team, p_ids, s_id):
    '''Compute the distance from the shooter to the nearest teammate.'''
    mask = (p_teams == s_team) & (p_ids != s_id)
    masked_dist = np.where(mask, dist_to_players, np.inf)
    min_dist = np.min(masked_dist, axis=1)
    return np.where(min_dist == np.inf, np.nan, min_dist)

def get_defenders_between_shooter_and_basket(p_x, p_y, s_x, s_y, p_teams, s_team, bx, by):
    '''Count defenders positioned between the shooter and the basket.'''
    
    # Vector Shooter -> Basket
    vec_b_x, vec_b_y = bx - s_x, by - s_y
    norm_b_sq = vec_b_x**2 + vec_b_y**2
    
    # Vector Shooter -> all players
    vec_p_x, vec_p_y = p_x - s_x, p_y - s_y
    
    # projection and distance
    dot = vec_p_x * vec_b_x + vec_p_y * vec_b_y
    proj_rel = dot / norm_b_sq
    
    perp_dist = np.sqrt((vec_p_x - proj_rel * vec_b_x)**2 + (vec_p_y - proj_rel * vec_b_y)**2)
    
    # filter: enemy player, between shoter and basket and close to connection line
    mask = (p_teams != s_team) & (proj_rel > 0) & (proj_rel < 1) & (perp_dist < 3)
    return np.sum(mask, axis=1)

def get_detect_screen(p_x, p_y, s_x, s_y, p_teams, s_team, p_ids, s_id, dist_to_players):
    '''Detect whether a teammate is setting a screen near the closest defender.'''
    
    # get closest defender
    def_mask = (p_teams != s_team)
    masked_def_dist = np.where(def_mask, dist_to_players, np.inf)
    nearest_def_idx = np.argmin(masked_def_dist, axis=1)
    
    # coords of closest defendet
    rows = np.arange(len(p_x))
    dx_near, dy_near = p_x[rows, nearest_def_idx][:, np.newaxis], p_y[rows, nearest_def_idx][:, np.newaxis]
    
    # vector shooter -> nearest defender
    line_x, line_y = dx_near - s_x, dy_near - s_y
    line_norm_sq = line_x**2 + line_y**2
    line_norm_sq[line_norm_sq == 0] = 1e-9
    
    # check teammates
    vec_p_x, vec_p_y = p_x - s_x, p_y - s_y
    dot = vec_p_x * line_x + vec_p_y * line_y
    proj_rel = dot / line_norm_sq
    
    perp_dist = np.sqrt((vec_p_x - proj_rel * line_x)**2 + (vec_p_y - proj_rel * line_y)**2)
    dist_to_def = np.sqrt((p_x - dx_near)**2 + (p_y - dy_near)**2)
    
    # screen criteria: teammate between shooter and defender and not too for to the defender
    tm_mask = (p_teams == s_team) & (p_ids != s_id)
    is_between = (proj_rel > 0) & (proj_rel < 1)
    screen_found = tm_mask & is_between & (perp_dist < 2) & (dist_to_def < 5)
    
    return np.any(screen_found, axis=1).astype(int)

def get_offensive_spacing(p_x, p_y, p_teams, s_team):
    '''Compute offensive spacing as mean distances between all offensive players'''
    
    spacing = []
    for i in range(len(p_x)):
        # filter for offensive players
        off_mask = (p_teams[i] == s_team[i])
        off_coords = np.stack([p_x[i, off_mask], p_y[i, off_mask]], axis=1)
        
        if len(off_coords) < 2:
            spacing.append(np.nan)
            continue
            
        # distances for all combinations
        dists = []
        for p1, p2 in combinations(off_coords, 2):
            dists.append(np.linalg.norm(p1 - p2))
        spacing.append(np.mean(dists))
        
    return np.array(spacing)

def get_teammate_between_defender(p_x, p_y, s_x, s_y, p_teams, s_team, p_ids, s_id, dist_to_players):
    '''
        Check if there is a teammate between the shooter and the next defending player.
        Similar to get screening player, but without min distance to defender
    '''
    
    def_mask = (p_teams != s_team)
    masked_def_dist = np.where(def_mask, dist_to_players, np.inf)
    nearest_def_idx = np.argmin(masked_def_dist, axis=1)
    
    rows = np.arange(len(p_x))
    dx_near, dy_near = p_x[rows, nearest_def_idx][:, np.newaxis], p_y[rows, nearest_def_idx][:, np.newaxis]
    
    line_x, line_y = dx_near - s_x, dy_near - s_y
    line_norm_sq = line_x**2 + line_y**2
    line_norm_sq[line_norm_sq == 0] = 1e-9
    
    vec_p_x, vec_p_y = p_x - s_x, p_y - s_y
    proj_rel = (vec_p_x * line_x + vec_p_y * line_y) / line_norm_sq
    perp_dist = np.sqrt((vec_p_x - proj_rel * line_x)**2 + (vec_p_y - proj_rel * line_y)**2)
    
    tm_mask = (p_teams == s_team) & (p_ids != s_id)
    between = tm_mask & (proj_rel > 0) & (proj_rel < 1) & (perp_dist < 5)
    
    return np.any(between, axis=1).astype(int)

def get_players_in_paint(p_x, p_y):
    '''Get the amount of players in the paint'''
    in_paint = (p_x > 94-19) & (p_x <= 94) & (p_y > 17) & (p_y < 33)
    return np.sum(in_paint, axis=1)

# Compute new features

In [11]:
df["nearest_teammate_distance"] = get_nearest_teammate_dist(dist_to_players, p_teams, s_team, p_ids, s_id)
df["defenders_between"] = get_defenders_between_shooter_and_basket(p_x, p_y, s_x, s_y, p_teams, s_team, basket_x, basket_y)
df["has_screen"] = get_detect_screen(p_x, p_y, s_x, s_y, p_teams, s_team, p_ids, s_id, dist_to_players)
df["offensive_spacing"] = get_offensive_spacing(p_x, p_y, p_teams, s_team)
df["teammate_between_defender"] = get_teammate_between_defender(p_x, p_y, s_x, s_y, p_teams, s_team, p_ids, s_id, dist_to_players)
df["players_in_paint"] = get_players_in_paint(p_x, p_y)

# Check und export

In [12]:
df.head()

,shooter_slot,shooter_x,shooter_y,shooter_team_id,shot_angle,distance_to_basket_tracking,nearest_defender_dist,avg_defender_dist,defenders_within_3ft,defenders_within_5ft,defenders_within_7ft,offensive_spacing_area,shooter_speed,player1_speed,player2_speed,player3_speed,player4_speed,player5_speed,player6_speed,player7_speed,player8_speed,player9_speed,player10_speed,defender_closing_speed,ball_height,ball_speed,ball_xy_speed,ACTION_TYPE,EVENTTIME,EVENT_TYPE,GAME_DATE,GAME_EVENT_ID,GAME_ID,GRID_TYPE,HTM,LOC_X,LOC_Y,MINUTES_REMAINING,PERIOD,PLAYER_ID,PLAYER_NAME,QUARTER,SECONDS_REMAINING,SHOT_ATTEMPTED_FLAG,SHOT_DISTANCE,SHOT_MADE_FLAG,SHOT_TIME,SHOT_TYPE,SHOT_ZONE_AREA,SHOT_ZONE_BASIC,SHOT_ZONE_RANGE,TEAM_ID,TEAM_NAME,VTM,tracking_game_clock,tracking_game_clock_old,event_GAME_ID,event_EVENTNUM,event_EVENTMSGTYPE,event_EVENTMSGACTIONTYPE,event_PERIOD,event_WCTIMESTRING,event_PCTIMESTRING,event_HOMEDESCRIPTION,event_NEUTRALDESCRIPTION,event_VISITORDESCRIPTION,event_SCORE,event_SCOREMARGIN,event_PERSON1TYPE,event_PLAYER1_ID,event_PLAYER1_NAME,event_PLAYER1_TEAM_ID,event_PLAYER1_TEAM_CITY,event_PLAYER1_TEAM_NICKNAME,event_PLAYER1_TEAM_ABBREVIATION,event_PERSON2TYPE,event_PLAYER2_ID,event_PLAYER2_NAME,event_PLAYER2_TEAM_ID,event_PLAYER2_TEAM_CITY,event_PLAYER2_TEAM_NICKNAME,event_PLAYER2_TEAM_ABBREVIATION,event_PERSON3TYPE,event_PLAYER3_ID,event_PLAYER3_NAME,event_PLAYER3_TEAM_ID,event_PLAYER3_TEAM_CITY,event_PLAYER3_TEAM_NICKNAME,event_PLAYER3_TEAM_ABBREVIATION,ball_x,ball_y,ball_z,player1_team_id,player1_id,player1_x,player1_y,player2_team_id,player2_id,player2_x,player2_y,player3_team_id,player3_id,player3_x,player3_y,player4_team_id,player4_id,player4_x,player4_y,player5_team_id,player5_id,player5_x,player5_y,player6_team_id,player6_id,player6_x,player6_y,player7_team_id,player7_id,player7_x,player7_y,player8_team_id,player8_id,player8_x,player8_y,player9_team_id,player9_id,player9_x,player9_y,player10_team_id,player10_id,player10_x,player10_y,shooter_x_t0,shooter_y_t0,defender1_dx_t0,defender1_dy_t0,defender1_dist_t0,defender2_dx_t0,defender2_dy_t0,defender2_dist_t0,defender3_dx_t0,defender3_dy_t0,defender3_dist_t0,defender4_dx_t0,defender4_dy_t0,defender4_dist_t0,defender5_dx_t0,defender5_dy_t0,defender5_dist_t0,attacker1_dx_t0,attacker1_dy_t0,attacker1_dist_t0,attacker2_dx_t0,attacker2_dy_t0,attacker2_dist_t0,attacker3_dx_t0,attacker3_dy_t0,attacker3_dist_t0,attacker4_dx_t0,attacker4_dy_t0,attacker4_dist_t0,shooter_x_t1,shooter_y_t1,defender1_dx_t1,defender1_dy_t1,defender1_dist_t1,defender2_dx_t1,defender2_dy_t1,defender2_dist_t1,defender3_dx_t1,defender3_dy_t1,defender3_dist_t1,defender4_dx_t1,defender4_dy_t1,defender4_dist_t1,defender5_dx_t1,defender5_dy_t1,defender5_dist_t1,attacker1_dx_t1,attacker1_dy_t1,attacker1_dist_t1,attacker2_dx_t1,attacker2_dy_t1,attacker2_dist_t1,attacker3_dx_t1,attacker3_dy_t1,attacker3_dist_t1,attacker4_dx_t1,attacker4_dy_t1,attacker4_dist_t1,shooter_x_t2,shooter_y_t2,defender1_dx_t2,defender1_dy_t2,defender1_dist_t2,defender2_dx_t2,defender2_dy_t2,defender2_dist_t2,defender3_dx_t2,defender3_dy_t2,defender3_dist_t2,defender4_dx_t2,defender4_dy_t2,defender4_dist_t2,defender5_dx_t2,defender5_dy_t2,defender5_dist_t2,attacker1_dx_t2,attacker1_dy_t2,attacker1_dist_t2,attacker2_dx_t2,attacker2_dy_t2,attacker2_dist_t2,attacker3_dx_t2,attacker3_dy_t2,attacker3_dist_t2,attacker4_dx_t2,attacker4_dy_t2,attacker4_dist_t2,shooter_x_t3,shooter_y_t3,defender1_dx_t3,defender1_dy_t3,defender1_dist_t3,defender2_dx_t3,defender2_dy_t3,defender2_dist_t3,defender3_dx_t3,defender3_dy_t3,defender3_dist_t3,defender4_dx_t3,defender4_dy_t3,defender4_dist_t3,defender5_dx_t3,defender5_dy_t3,defender5_dist_t3,attacker1_dx_t3,attacker1_dy_t3,attacker1_dist_t3,attacker2_dx_t3,attacker2_dy_t3,attacker2_dist_t3,attacker3_dx_t3,attacker3_dy_t3,attacker3_dist_t3,attacker4_dx_t3,attacker4_dy_t3,attacker4_dist_t3,shooter_x_t4,shooter_y_t4,defender1_dx_t4,defender1_dy_t4,defender1_dist_t4,defender2_dx_t4,defender2_dy_t4,defender2_d

In [13]:
df.info(max_cols=350, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 84103 entries, 0 to 84464
Data columns (total 312 columns):
 #    Column                           Non-Null Count  Dtype  
---   ------                           --------------  -----  
 0    shooter_slot                     84103 non-null  float64
 1    shooter_x                        84103 non-null  float64
 2    shooter_y                        84103 non-null  float64
 3    shooter_team_id                  84103 non-null  float64
 4    shot_angle                       84103 non-null  float64
 5    distance_to_basket_tracking      84103 non-null  float64
 6    nearest_defender_dist            84103 non-null  float64
 7    avg_defender_dist                84103 non-null  float64
 8    defenders_within_3ft             84103 non-null  float64
 9    defenders_within_5ft             84103 non-null  float64
 10   defenders_within_7ft             84103 non-null  float64
 11   offensive_spacing_area           84103 non-null  float64
 12   shooter

In [14]:
df.describe()

,shooter_slot,shooter_x,shooter_y,shooter_team_id,shot_angle,distance_to_basket_tracking,nearest_defender_dist,avg_defender_dist,defenders_within_3ft,defenders_within_5ft,defenders_within_7ft,offensive_spacing_area,shooter_speed,player1_speed,player2_speed,player3_speed,player4_speed,player5_speed,player6_speed,player7_speed,player8_speed,player9_speed,player10_speed,defender_closing_speed,ball_height,ball_speed,ball_xy_speed,EVENTTIME,GAME_DATE,GAME_EVENT_ID,GAME_ID,LOC_X,LOC_Y,MINUTES_REMAINING,PERIOD,PLAYER_ID,QUARTER,SECONDS_REMAINING,SHOT_ATTEMPTED_FLAG,SHOT_DISTANCE,SHOT_MADE_FLAG,SHOT_TIME,TEAM_ID,tracking_game_clock,tracking_game_clock_old,event_GAME_ID,event_EVENTNUM,event_EVENTMSGTYPE,event_EVENTMSGACTIONTYPE,event_PERIOD,event_NEUTRALDESCRIPTION,event_PERSON1TYPE,event_PLAYER1_ID,event_PLAYER1_TEAM_ID,event_PERSON2TYPE,event_PLAYER2_ID,event_PLAYER2_TEAM_ID,event_PERSON3TYPE,event_PLAYER3_ID,event_PLAYER3_TEAM_ID,ball_x,ball_y,ball_z,player1_team_id,player1_id,player1_x,player1_y,player2_team_id,player2_id,player2_x,player2_y,player3_team_id,player3_id,player3_x,player3_y,player4_team_id,player4_id,player4_x,player4_y,player5_team_id,player5_id,player5_x,player5_y,player6_team_id,player6_id,player6_x,player6_y,player7_team_id,player7_id,player7_x,player7_y,player8_team_id,player8_id,player8_x,player8_y,player9_team_id,player9_id,player9_x,player9_y,player10_team_id,player10_id,player10_x,player10_y,shooter_x_t0,shooter_y_t0,defender1_dx_t0,defender1_dy_t0,defender1_dist_t0,defender2_dx_t0,defender2_dy_t0,defender2_dist_t0,defender3_dx_t0,defender3_dy_t0,defender3_dist_t0,defender4_dx_t0,defender4_dy_t0,defender4_dist_t0,defender5_dx_t0,defender5_dy_t0,defender5_dist_t0,attacker1_dx_t0,attacker1_dy_t0,attacker1_dist_t0,attacker2_dx_t0,attacker2_dy_t0,attacker2_dist_t0,attacker3_dx_t0,attacker3_dy_t0,attacker3_dist_t0,attacker4_dx_t0,attacker4_dy_t0,attacker4_dist_t0,shooter_x_t1,shooter_y_t1,defender1_dx_t1,defender1_dy_t1,defender1_dist_t1,defender2_dx_t1,defender2_dy_t1,defender2_dist_t1,defender3_dx_t1,defender3_dy_t1,defender3_dist_t1,defender4_dx_t1,defender4_dy_t1,defender4_dist_t1,defender5_dx_t1,defender5_dy_t1,defender5_dist_t1,attacker1_dx_t1,attacker1_dy_t1,attacker1_dist_t1,attacker2_dx_t1,attacker2_dy_t1,attacker2_dist_t1,attacker3_dx_t1,attacker3_dy_t1,attacker3_dist_t1,attacker4_dx_t1,attacker4_dy_t1,attacker4_dist_t1,shooter_x_t2,shooter_y_t2,defender1_dx_t2,defender1_dy_t2,defender1_dist_t2,defender2_dx_t2,defender2_dy_t2,defender2_dist_t2,defender3_dx_t2,defender3_dy_t2,defender3_dist_t2,defender4_dx_t2,defender4_dy_t2,defender4_dist_t2,defender5_dx_t2,defender5_dy_t2,defender5_dist_t2,attacker1_dx_t2,attacker1_dy_t2,attacker1_dist_t2,attacker2_dx_t2,attacker2_dy_t2,attacker2_dist_t2,attacker3_dx_t2,attacker3_dy_t2,attacker3_dist_t2,attacker4_dx_t2,attacker4_dy_t2,attacker4_dist_t2,shooter_x_t3,shooter_y_t3,defender1_dx_t3,defender1_dy_t3,defender1_dist_t3,defender2_dx_t3,defender2_dy_t3,defender2_dist_t3,defender3_dx_t3,defender3_dy_t3,defender3_dist_t3,defender4_dx_t3,defender4_dy_t3,defender4_dist_t3,defender5_dx_t3,defender5_dy_t3,defender5_dist_t3,attacker1_dx_t3,attacker1_dy_t3,attacker1_dist_t3,attacker2_dx_t3,attacker2_dy_t3,attacker2_dist_t3,attacker3_dx_t3,attacker3_dy_t3,attacker3_dist_t3,attacker4_dx_t3,attacker4_dy_t3,attacker4_dist_t3,shooter_x_t4,shooter_y_t4,defender1_dx_t4,defender1_dy_t4,defender1_dist_t4,defender2_dx_t4,defender2_dy_t4,defender2_dist_t4,defender3_dx_t4,defender3_dy_t4,defender3_dist_t4,defender4_dx_t4,defender4_dy_t4,defender4_dist_t4,defender5_dx_t4,defender5_dy_t4,defender5_dist_t4,attacker1_dx_t4,attacker1_dy_t4,attacker1_dist_t4,attacker2_dx_t4,attacker2_dy_t4,attacker2_dist_t4,attacker3_dx_t4,attacker3_dy_t4,attacker3_dist_t4,attacker4_dx_t4,attacker4_dy_t4,attacker4_dist_t4,shooter_x_t5,shooter_y_t5,defender1_dx_t5,defender1_dy_t5,defender1_dist_t5,defender2_dx_t5,defender2_dy_t5,defender2_dist_t5,defender3_dx_t5,defender3_dy_t5,defender3_dist_t5,defender4_d

In [15]:
#df.to_csv('merged_filtered_prep_20_players_v3.csv')
df.to_csv('merged_filtered_prep_all_players_v3.csv')

In [16]:
#df = pd.read_csv('merged_filtered_prep_20_players_v3.csv')
df = pd.read_csv('merged_filtered_prep_all_players_v3.csv')

In [17]:
df.shape

(84103, 313)

## Train test split of new and old data

In [18]:
# Load old dataset to make the same datasplit on both datasets
from app.data_providers import test_train_dataset
data = test_train_dataset()
X_train = data['train'].drop('SHOT_MADE_FLAG', axis=1)
y_train = data['train']['SHOT_MADE_FLAG']

X_test = data['test'].drop('SHOT_MADE_FLAG', axis=1)
y_test = data['test']['SHOT_MADE_FLAG']

df_old = pd.concat([data['train'], data['test']], axis=0)

Repo card metadata block was not found. Setting CardData to empty.


In [19]:
# GAME_DATE to year, for filtering
df_old['GAME_DATE'] = pd.to_datetime(df_old['GAME_DATE'].astype(int).astype(str),format='%Y%m%d')
df_old['year'] = df_old['GAME_DATE'].dt.year
df_old.head()

,GAME_ID_x,GAME_EVENT_ID,SHOT_TYPE,SHOT_DISTANCE,SHOT_ZONE_RANGE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,LOC_X,LOC_Y,ACTION_TYPE,HTM,VTM,GAME_DATE,TEAM_ID,PLAYER1_TEAM_ABBREVIATION,PLAYER_ID,scoreHome,scoreAway,shotResult,MINUTES_REMAINING,SECONDS_REMAINING,clock,PLAYER_NAME,PERIOD_x,PLAYER2_ID,PLAYER2_NAME,PLAYER2_TEAM_ABBREVIATION,PLAYER3_ID,PLAYER3_NAME,is_playoffs,PCTIMESTRING,SHOT_MADE_FLAG,IS_HOME,points,pointsHome,pointsAway,scoreHomeBeforeShot,scoreAwayBeforeShot,scoreMargin,scoreMarginBeforeShot,TimeRemainingInPeriod,TotalPlayedTime,TimeRemainingInGame,IsOvertime,OvertimeNumber,IsClutchTime,OPPONENT_INTERFERED,ANGLE,ANGLE_SECTOR,ABS_ANGLE,ANGLE_SIN,ANGLE_COS,MAIN_ACTION_TYPE,__index_level_0__,year
0,29600008.0,281.0,2PT Field Goal,8.0,8-16 ft.,In The Paint (Non-RA),Left Side(L),-42.0,69.0,Jump Shot,MIN,SAS,1996-11-01,1.610613e+09,MIN,708.0,50,44,Missed,5.0,25.0,PT05M25.00S,Kevin Garnett,3.0,0.0,None,None,0.0,None,False,5:25,0.0,1,2,0,0,50.0,44.0,6,6.0,325,1835,1045,0,0,0,False,-31.328693,0,31.328693,0.087123,0.996198,Jump,448756,1996
1,29600008.0,168.0,1PT Free Throw,15.0,8-16 ft.,Mid-Range,Center(C),0.0,15.0,Free Throw,MIN,SAS,1996-11-01,1.610613e+09,MIN,708.0,27,31,Missed,6.0,31.0,PT06M31.00S,Kevin Garnett,2.0,0.0,None,None,0.0,None,False,6:31,0.0,1,1,0,0,27.0,31.0,-4,-4.0,391,1049,1831,0,0,0,False,0.000000,0,0.000000,0.000000,1.000000,Other,448749,1996
2,29600008.0,404.0,2PT Field Goal,12.0,8-16 ft.,Mid-Range,Right Side(R),112.0,58.0,Jump Shot,MIN,SAS,1996-11-01,1.610613e+09,MIN,708.0,71,70,Missed,2.0,53.0,PT02M53.00S,Kevin Garnett,4.0,0.0,None,None,0.0,None,False,2:53,0.0,1,2,0,0,71.0,70.0,1,1.0,173,2707,173,0,0,1,False,62.622297,1,62.622297,-0.208025,0.978123,Jump,448762,1996
3,29600008.0,206.0,2PT Field Goal,14.0,8-16 ft.,Mid-Range,Right Side(R),140.0,-5.0,Jump Shot,MIN,SAS,1996-11-01,1.610613e+09,MIN,708.0,35,34,Missed,3.0,13.0,PT03M13.00S,Kevin Garnett,2.0,0.0,None,None,0.0,None,False,3:13,0.0,1,2,0,0,35.0,34.0,1,1.0,193,1247,1633,0,0,0,False,92.045408,2,92.045408,-0.807099,-0.590417,Jump,448751,1996
4,29600008.0,260.0,2PT Field Goal,16.0,16-24 ft.,Mid-Range,Right Side(R),151.0,67.0,Jump Shot,MIN,SAS,1996-11-01,1.610613e+09,MIN,708.0,48,42,Missed,7.0,45.0,PT07M45.00S,Kevin Garnett,3.0,0.0,None,None,0.0,None,False,7:45,0.0,1,2,0,0,48.0,42.0,6,6.0,465,1695,1185,0,0,0,False,66.072727,1,66.072727,-0.099118,-0.995076,Jump,448754,1996


In [20]:
def create_match_key(df, id_cols):
    '''Create a key to match old and new games'''

    match_col = df[id_cols[0]].astype(int).astype(str) + "_"
    
    for id_col in id_cols[1:]:
        match_col = match_col + df[id_col].astype(int).astype(str) + "_"

    return match_col

df_old["match_key"] = create_match_key(df_old, ['GAME_ID_x', 'GAME_EVENT_ID'])
df["match_key"] = create_match_key(df, ['GAME_ID', 'GAME_EVENT_ID'])


old_keys = set(df_old["match_key"])
new_keys = set(df["match_key"])

intersection = old_keys & new_keys

print("OLD events:", len(old_keys))
print("NEW events:", len(new_keys))
print("OVERLAP:", len(intersection))

coverage_old = len(intersection) / len(old_keys)
coverage_new = len(intersection) / len(new_keys)

print("Coverage old -> new:", coverage_old)
print("Coverage new -> old:", coverage_new)

missing_keys = old_keys - new_keys
missing_df = df_old[df_old["match_key"].isin(missing_keys)]

OLD events: 537909
NEW events: 84103
OVERLAP: 8149
Coverage old -> new: 0.01514940259411908
Coverage new -> old: 0.09689309537115204


# Filter old and new dataframes by overlapping columns and make the same train test split for both

In [24]:
from sklearn.model_selection import train_test_split

In [25]:
if do_filter_for_players:
    df_old["match_key"] = create_match_key(df_old, ['GAME_ID_x', 'GAME_EVENT_ID'])
    df["match_key"] = create_match_key(df, ['GAME_ID', 'GAME_EVENT_ID'])
else:
    # Only based on game id, since old data was filtered by players
    df_old["match_key"] = create_match_key(df_old, ['GAME_ID_x'])
    df["match_key"] = create_match_key(df, ['GAME_ID'])

common_uids = set(df_old["match_key"]).intersection(set(df["match_key"]))

df_old_filt = df_old[df_old["match_key"].isin(common_uids)].copy().sort_values("match_key").reset_index(drop=True)
df_new_filt = df[df["match_key"].isin(common_uids)].copy().sort_values("match_key").reset_index(drop=True)

In [26]:
df_new_filt.shape

(57510, 314)

In [27]:
# Make the same split on old and new data
uids = df_old["match_key"].drop_duplicates()

train_uids, test_uids = train_test_split(
    uids,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_old = df_old_filt[df_old_filt["match_key"].isin(train_uids)].copy()
test_old  = df_old_filt[df_old_filt["match_key"].isin(test_uids)].copy()

train_new = df_new_filt[df_new_filt["match_key"].isin(train_uids)].copy()
test_new  = df_new_filt[df_new_filt["match_key"].isin(test_uids)].copy()

#train_old.to_parquet("train_old_v3.parquet", index=False)
#test_old.to_parquet("test_old_v3.parquet", index=False)

#train_new.to_parquet("train_new_20_players_v3.parquet", index=False)
#test_new.to_parquet("test_new_20_players_v3.parquet", index=False)

train_new.to_parquet("train_new_all_players_v3.parquet", index=False)
test_new.to_parquet("test_new_all_players_v3.parquet", index=False)



In [28]:
train_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44403 entries, 354 to 57509
Columns: 314 entries, Unnamed: 0 to match_key
dtypes: float64(257), int64(27), object(30)
memory usage: 106.7+ MB
